<a href="https://colab.research.google.com/github/imgeG/CYT-300-Group-1/blob/M5%266/CYT300_M4_imge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup

In [1]:
import re, os, sys, glob, mailbox, urllib.request
import numpy as np
import pandas as pd
import joblib
import kagglehub
from email import policy
from email.parser import BytesParser
from scipy.sparse import hstack, csr_matrix, save_npz
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold, cross_validate, cross_val_score
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/CYT300/imge'     # where all outputs are saved
os.makedirs(OUT, exist_ok=True)
CACHE = f'{OUT}/cache'                          # saved raw emails, so datasets download only once
os.makedirs(CACHE, exist_ok=True)

sys.path.append(OUT)
from features import *                          # cleaning + feature functions (shared with the milter)

COLS = ['sender', 'receiver', 'subject', 'body', 'label']   # columns every dataset is converted to
KEY  = ['sender', 'receiver', 'subject', 'body']            # columns that define "the same email"
SEED = 42
pd.set_option('display.width', 200)             # print wide tables without '...'
pd.set_option('display.max_columns', 20)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Load training data

In [2]:
# ---------- Email-file readers (used for Nazario mailboxes and phishing_pot .eml files) ----------
def read_eml(path):
    """Read one .eml file."""
    try:
        with open(path, 'rb') as f:
            return msg_to_row(BytesParser(policy=policy.default).parse(f))
    except Exception:
        return None

def read_mbox(path):
    """Read every email in an mbox mailbox file, skipping any that can't be parsed."""
    with open(path, 'rb') as f:
        if not f.read(5).startswith(b'From '):                    # mbox files start with "From "
            raise ValueError(f"{path} is not an mbox file")
    box = mailbox.mbox(path, factory=lambda f: BytesParser(policy=policy.default).parse(f), create=False)
    rows = []
    for key in box.iterkeys():
        try:
            r = msg_to_row(box[key])
        except Exception:
            r = None
        if r:
            rows.append(r)
    return rows

# Keep only English emails (the training data is mostly English)
EN_RE = r'\b(?:the|and|you|your|to|of|for|is|this|please|with)\b'
def english(d):
    """Keep emails containing at least 5 common English words."""
    return d[(d['subject'] + ' ' + d['body']).str.lower().str.count(EN_RE) >= 5]

In [3]:
# ---------- Training data: Kaggle + Nazario 2015-2025 (downloaded once, then loaded from cache) ----------
TRAIN_RAW = f'{CACHE}/train_raw.parquet'

if os.path.exists(TRAIN_RAW):
    df = pd.read_parquet(TRAIN_RAW)
    print(f"Loaded cached training data: {len(df)} emails")
else:
    # Kaggle phishing dataset (Enron and Ling excluded: no sender/receiver addresses)
    path = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")
    FILES = {"CEAS_08": "CEAS_08.csv", "Nazario": "Nazario.csv",
             "Nigerian_Fraud": "Nigerian_Fraud.csv", "SpamAssassin": "SpamAssasin.csv"}
    df = pd.concat([pd.read_csv(os.path.join(path, f))[COLS].assign(source=name)
                    for name, f in FILES.items()], ignore_index=True)
    print(f"Kaggle datasets: {len(df)} emails")

    # Modern phishing: Nazario Phishing Corpus 2015-2025 (CC BY 4.0) - https://monkey.org/~jose/phishing/
    NAZ_DIR = '/content/nazario'
    os.makedirs(NAZ_DIR, exist_ok=True)
    rows = []
    for y in range(2015, 2026):
        fn = f'{NAZ_DIR}/phishing-{y}'
        if not os.path.exists(fn):
            urllib.request.urlretrieve(f'https://monkey.org/~jose/phishing/phishing-{y}', fn)
        year_rows = read_mbox(fn)
        rows += year_rows
        print(f"Nazario {y}: {len(year_rows)} emails")
    naz_new = english(pd.DataFrame(rows)).assign(label=1, source='Nazario_2015_2025')
    df = pd.concat([df, naz_new[COLS + ['source']]], ignore_index=True)
    print(f"Added {len(naz_new)} modern phishing emails -> {len(df)} training emails before cleaning")

    df.to_parquet(TRAIN_RAW, index=False)
    print("Saved to cache:", TRAIN_RAW)

Using Colab cache for faster access to the 'phishing-email-dataset' dataset.
Kaggle datasets: 49860 emails
Nazario 2015: 307 emails
Nazario 2016: 494 emails
Nazario 2017: 325 emails
Nazario 2018: 288 emails
Nazario 2019: 242 emails
Nazario 2020: 155 emails
Nazario 2021: 101 emails
Nazario 2022: 247 emails
Nazario 2023: 418 emails
Nazario 2024: 403 emails
Nazario 2025: 481 emails
Added 2910 modern phishing emails -> 52770 training emails before cleaning
Saved to cache: /content/drive/MyDrive/CYT300/imge/cache/train_raw.parquet


## 3. Build training features

In [4]:
train = build_features(clean(df)).reset_index(drop=True)
y_train = train['label'].values
print(f"Training emails: {len(train)} | phishing ratio: {y_train.mean():.3f}")
print(train.groupby('source')['label'].agg(['count', 'mean']).round(3))

# Remove features that never change in this data (no information)
constant = [f for f in NUM_FEATURES if train[f].nunique() <= 1]
NUM_FEATURES = [f for f in NUM_FEATURES if f not in constant]
print("Removed constant features:", constant)
print("Rare features (<0.1% of emails):", [f for f in NUM_FEATURES if (train[f] != 0).mean() < 0.001])

clean(): 52770 -> 52277 rows
build_features(): dropped 1064 rows with no valid sender address
build_features(): dropped 0 rows with no text after cleaning
Training emails: 51213 | phishing ratio: 0.582
                   count   mean
source                         
CEAS_08            38183  0.547
Nazario             1531  1.000
Nazario_2015_2025   2703  1.000
Nigerian_Fraud      2997  1.000
SpamAssassin        5799  0.295
Removed constant features: ['s_leet', 's_nonascii']
Rare features (<0.1% of emails): ['s_typosquat', 's_brand_in_subdomain', 's_punycode', 'u_at']


## 4. Fit encoders (training data only)

In [5]:
# ----- One-hot encoding: 10 most common sender TLDs, everything else -> TLD_other -----
TOP_TLDS = sorted(train['sender_tld'].value_counts().head(10).index)

def encode_tld(d):
    out = pd.DataFrame({f'TLD_{t}': (d['sender_tld'] == t).astype(int) for t in TOP_TLDS}, index=d.index)
    out['TLD_other'] = (~d['sender_tld'].isin(TOP_TLDS)).astype(int)
    return out

def numeric_matrix(d):
    """Numeric features + TLD columns, in a fixed column order.
    (Sender-domain reputation was removed: it did not generalize to unseen data.)"""
    return pd.concat([d[NUM_FEATURES], encode_tld(d)], axis=1)

X_num = numeric_matrix(train)
FEATURES = list(X_num.columns)

# ----- Scale numeric features to 0-1 (same range as TF-IDF) -----
scaler = MinMaxScaler().fit(X_num)

# ----- TF-IDF on subject and body -----
# Words that only identify which dataset an email came from -> removed so the model can't use them
CORPUS_WORDS = {'python', 'perl', 'opensuse', 'linux', 'dev', 'mailing', 'list', 'lists', 'unsubscribe',
                'listinfo', 'mailman', 'spamassassin', 'apache', 'sourceforge', 'ceas', 'spam',
                'monkey', 'jose', 'nazario', 'cnn', 'lllp', 'utf', 'iso',
                'spambayes', 'uai', 'rant', 'fork', 'rpm', 'tony', 'meyer'}
STOP  = list(ENGLISH_STOP_WORDS | CORPUS_WORDS)
TOKEN = r'(?u)\b[a-z][a-z]+\b'                     # words of 2+ letters (numbers ignored)

text_tf = ColumnTransformer([
    ('subj', TfidfVectorizer(ngram_range=(1, 2), min_df=10, max_features=2000, sublinear_tf=True,
                             stop_words=STOP, token_pattern=TOKEN), 'subject_clean'),
    ('body', TfidfVectorizer(ngram_range=(1, 2), min_df=20, max_df=0.9, max_features=5000, sublinear_tf=True,
                             stop_words=STOP, token_pattern=TOKEN), 'body_clean'),
]).fit(train)

## 5. Build and save the training matrix

In [6]:
X_train = hstack([csr_matrix(scaler.transform(X_num)), text_tf.transform(train)]).tocsr()
FULL_FEATURE_NAMES = FEATURES + list(text_tf.get_feature_names_out())
n_num = len(FEATURES)                                     # numeric columns come first in X
print("X_train:", X_train.shape)

save_npz(f'{OUT}/X_train.npz', X_train)
np.save(f'{OUT}/y_train.npy', y_train)
np.save(f'{OUT}/groups_train.npy', train['sender_addr'].values)   # sender per row, for grouped validation

TRAIN_KEYS = set(pd.util.hash_pandas_object(train[KEY], index=False))   # fingerprints of training emails
joblib.dump({'scaler': scaler, 'text_tf': text_tf, 'top_tlds': TOP_TLDS,
             'num_features': NUM_FEATURES, 'features': FEATURES, 'full_feature_names': FULL_FEATURE_NAMES,
             'strong_rules': STRONG, 'corpus_words': CORPUS_WORDS, 'train_keys': TRAIN_KEYS},
            f'{OUT}/preprocessing.joblib')
print("Saved training files to", OUT)

X_train: (51213, 7043)
Saved training files to /content/drive/MyDrive/CYT300/imge


## 6. External test set (sources never seen in training)
- **Phishing:** phishing_pot (real phishing from honeypot inboxes, 2022-now, CC BY-NC 4.0)
- **Legit:** Enron (real corporate email, version with addresses)

In [7]:
def prepare_test(raw):
    """Raw emails -> (X_test, y_test, feature table) using the tools fitted on the training data."""
    d = build_features(clean(raw[COLS]))
    overlap = pd.util.hash_pandas_object(d[KEY], index=False).isin(list(TRAIN_KEYS)).values
    print(f"prepare_test(): removed {overlap.sum()} emails already in the training data")
    d = d[~overlap].reset_index(drop=True)
    if d.empty:
        raise ValueError("No test emails left: every email is also in the training data.")
    X = hstack([csr_matrix(scaler.transform(numeric_matrix(d)[FEATURES])), text_tf.transform(d)]).tocsr()
    return X, d['label'].values, d

N_PER_CLASS = 3000
TEST_RAW = f'{CACHE}/test_raw.parquet'

if os.path.exists(TEST_RAW):
    test_raw = pd.read_parquet(TEST_RAW)
    print(f"Loaded cached test data: {len(test_raw)} emails")
else:
    os.system('pip -q install datasets')
    from datasets import load_dataset

    # Phishing: phishing_pot (about 2.5 GB, only downloaded when the cache is missing)
    if not os.path.exists('/content/phishing_pot'):
        os.system('git clone --depth 1 -q https://github.com/rf-peixoto/phishing_pot /content/phishing_pot')
    files = sorted(glob.glob('/content/phishing_pot/email/*.eml'))
    phish = pd.DataFrame([r for r in map(read_eml, files) if r])
    print(f"phishing_pot: parsed {len(phish)} of {len(files)} emails")
    # The repo hides recipients as 'phishing@pot'; replace with a neutral address so receiver features stay fair
    for c in ['sender', 'receiver', 'body']:
        phish[c] = phish[c].str.replace('phishing@pot', 'user@example.com', regex=False)

    # Legit: Enron
    enron = (load_dataset('corbt/enron-emails', split='train')
             .shuffle(seed=SEED).select(range(N_PER_CLASS * 3)).to_pandas())
    as_text = lambda v: ', '.join(map(str, v)) if isinstance(v, (list, tuple, np.ndarray)) else ('' if v is None else str(v))
    legit = pd.DataFrame({out_c: enron[in_c].map(as_text) for out_c, in_c in
                          [('sender', 'from'), ('receiver', 'to'), ('subject', 'subject'), ('body', 'body')]})

    phish, legit = english(phish), english(legit)
    print(f"English emails available: {len(phish)} phishing, {len(legit)} legit")
    n = min(N_PER_CLASS, len(phish), len(legit))
    test_raw = pd.concat([phish.sample(n, random_state=SEED).assign(label=1),
                          legit.sample(n, random_state=SEED).assign(label=0)], ignore_index=True)

    test_raw.to_parquet(TEST_RAW, index=False)
    print("Saved to cache:", TEST_RAW)

X_test, y_test, d_test = prepare_test(test_raw)
print("X_test:", X_test.shape, "| phishing ratio:", round(y_test.mean(), 3))
save_npz(f'{OUT}/X_test.npz', X_test)
np.save(f'{OUT}/y_test.npy', y_test)
np.save(f'{OUT}/groups_test.npy', d_test['sender_addr'].values)              # sender per test row (for M6)
np.save(f'{OUT}/rules_test.npy', (d_test[STRONG] > 0).any(axis=1).values)    # did a strong rule fire (for M6)
print("Saved test files to", OUT)

phishing_pot: parsed 12268 of 12271 emails


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  185MB            

data/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  167MB            

data/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  141MB            

data/train-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/517401 [00:00<?, ? examples/s]

English emails available: 7251 phishing, 7321 legit
Saved to cache: /content/drive/MyDrive/CYT300/imge/cache/test_raw.parquet
clean(): 6000 -> 5939 rows
build_features(): dropped 235 rows with no valid sender address
build_features(): dropped 0 rows with no text after cleaning
prepare_test(): removed 0 emails already in the training data
X_test: (5704, 7043) | phishing ratio: 0.481
Saved test files to /content/drive/MyDrive/CYT300/imge


## 7. Sanity and leakage checks

In [8]:
def check(name, cond, detail=''):
    print(f"[{'PASS' if cond else 'FAIL'}] {name}" + ('' if cond else f"  -> {detail}"))

check("train/test have the same columns", X_train.shape[1] == X_test.shape[1] == len(FULL_FEATURE_NAMES))
check("no NaN/inf values", np.isfinite(X_train.data).all() and np.isfinite(X_test.data).all())
check("test rows line up with labels", len(d_test) == X_test.shape[0] and (d_test['label'].values == y_test).all())
check("no exact train/test duplicates", not pd.util.hash_pandas_object(d_test[KEY], index=False).isin(list(TRAIN_KEYS)).any())
shared = d_test['sender_addr'].isin(set(train['sender_addr'])).mean()
check("test senders unseen in training (<5% shared)", shared < 0.05, f"{shared:.1%} shared")
check("brand check: no false matches (purchase.com, pineapple.com, x.onmicrosoft.com)",
      not any(domain_risk('purchase.com')) and not any(domain_risk('pineapple.com'))
      and not any(domain_risk('abc.onmicrosoft.com')))
check("brand check: real lookalikes still caught (paypa1, paypall, paypal-support)",
      domain_risk('paypa1.com')[1] == 1 and any(domain_risk('paypall.com')) and domain_risk('paypal-support.com')[2] == 1)

# Shuffled-label test: with random labels the model must score ~0.5 AUC, otherwise something leaks
auc = cross_val_score(LogisticRegression(max_iter=1000), X_train, np.random.RandomState(0).permutation(y_train),
                      groups=train['sender_addr'], cv=GroupKFold(3), scoring='roc_auc').mean()
check("shuffled labels give AUC ~0.5", abs(auc - 0.5) < 0.03, f"AUC {auc:.3f}")

prep = joblib.load(f'{OUT}/preprocessing.joblib')
idx = np.random.RandomState(SEED).choice(len(d_test), 50, replace=False)
bad = 0
for i in idx:
    r = d_test.iloc[i]
    x, _ = featurize_one(r['sender'], r['receiver'], r['subject'], r['body'], prep)
    if x is None or abs(x - X_test[i]).max() > 1e-9:
        bad += 1
check("single-email path matches batch features (50 emails)", bad == 0, f"{bad} mismatches")

[PASS] train/test have the same columns
[PASS] no NaN/inf values
[PASS] test rows line up with labels
[PASS] no exact train/test duplicates
[PASS] test senders unseen in training (<5% shared)
[PASS] brand check: no false matches (purchase.com, pineapple.com, x.onmicrosoft.com)
[PASS] brand check: real lookalikes still caught (paypa1, paypall, paypal-support)
[PASS] shuffled labels give AUC ~0.5
[PASS] single-email path matches batch features (50 emails)


## 8. Which feature set generalizes best?
Same model, three feature sets. CV = grouped validation on training data; Test = unseen external set.

In [9]:
sets = {'A: engineered only': (X_train[:, :n_num], X_test[:, :n_num]),
        'B: TF-IDF only':     (X_train[:, n_num:], X_test[:, n_num:]),
        'C: both':            (X_train,            X_test)}

rows = []
for name, (Xtr, Xte) in sets.items():
    model = LogisticRegression(max_iter=2000, C=1.0)
    cv = cross_validate(model, Xtr, y_train, groups=train['sender_addr'], cv=GroupKFold(5), scoring='f1', n_jobs=-1)
    pred = model.fit(Xtr, y_train).predict(Xte)
    rows.append({'experiment': name, 'CV F1': cv['test_score'].mean(), 'Test F1': f1_score(y_test, pred),
                 'Test precision': precision_score(y_test, pred), 'Test recall': recall_score(y_test, pred)})
print(pd.DataFrame(rows).set_index('experiment').round(3))

                    CV F1  Test F1  Test precision  Test recall
experiment                                                     
A: engineered only  0.922    0.822           0.769        0.884
B: TF-IDF only      0.979    0.813           0.746        0.892
C: both             0.983    0.828           0.752        0.921


## 9. How often do the phishing indicators fire? (% of emails)
Shows the era gap: modern phishing uses tricks the 2008 training data barely contains.

In [10]:
IND = ['s_typosquat', 's_leet', 's_brand_in_domain', 's_brand_in_subdomain', 's_display_brand_mismatch',
       's_punycode', 's_susp_tld', 'u_ip', 'u_shortener', 'u_at', 'u_anchor_mismatch', 'n_invisible', 'mixed_script']
fires = lambda d: (d[IND] > 0).mean() * 100
print(pd.DataFrame({'train phishing': fires(train[train['label'] == 1]), 'train legit': fires(train[train['label'] == 0]),
                    'test phishing':  fires(d_test[d_test['label'] == 1]), 'test legit':  fires(d_test[d_test['label'] == 0])}).round(2))

                          train phishing  train legit  test phishing  test legit
s_typosquat                         0.06         0.00           0.00        0.00
s_leet                              0.00         0.00           0.00        0.00
s_brand_in_domain                   0.08         0.28           0.00        0.00
s_brand_in_subdomain                0.05         0.00           0.00        0.03
s_display_brand_mismatch            1.88         0.05           5.97        0.00
s_punycode                          0.00         0.00           0.00        0.00
s_susp_tld                          1.14         0.29           0.98        0.00
u_ip                                0.48         0.10           3.82        0.34
u_shortener                         0.24         0.31           5.35        0.00
u_at                                0.09         0.02           0.18        0.03
u_anchor_mismatch                   0.69         0.00           5.46        0.03
n_invisible                 

## 10. Recall at fixed false-alarm rates: ML vs ML + rules
The test set is split into two balanced halves by sender: the **validation half** picks the threshold, the **final half** gives the reported numbers.

In [11]:
sgkf = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=SEED)
val_idx, fin_idx = next(sgkf.split(X_test, y_test, groups=d_test['sender_addr']))
yv, yf = y_test[val_idx], y_test[fin_idx]
print(f"validation: {len(yv)} emails ({yv.mean():.0%} phishing) | final: {len(yf)} emails ({yf.mean():.0%} phishing)")

rules = (d_test[STRONG] > 0).any(axis=1).values             # True if any strong indicator fires
mdl   = LogisticRegression(C=10, max_iter=3000).fit(X_train, y_train)
p     = mdl.predict_proba(X_test)[:, 1]                     # phishing probability per test email
print(f"ROC-AUC on final half: {roc_auc_score(yf, p[fin_idx]):.3f}")

def summarize(target, system, pred):
    pred = pred.astype(int)
    return {'target false-alarm rate': f"{target:.0%}", 'system': system,
            'phishing caught (recall)': recall_score(yf, pred),
            'false-alarm rate': ((pred == 1) & (yf == 0)).sum() / (yf == 0).sum(),
            'precision': precision_score(yf, pred, zero_division=0), 'F1': f1_score(yf, pred)}

rows = [summarize(0, 'rules only', rules[fin_idx])]
for target in [0.01, 0.05, 0.10]:
    thr = np.quantile(p[val_idx][yv == 0], 1 - target)       # threshold giving that rate on validation legit mail
    rows.append(summarize(target, 'ML', p[fin_idx] >= thr))
    rows.append(summarize(target, 'ML OR rules', (p[fin_idx] >= thr) | rules[fin_idx]))
print(pd.DataFrame(rows).round(3).to_string(index=False))

validation: 2970 emails (49% phishing) | final: 2734 emails (47% phishing)
ROC-AUC on final half: 0.926
target false-alarm rate      system  phishing caught (recall)  false-alarm rate  precision    F1
                     0%  rules only                     0.162             0.003      0.977 0.278
                     1%          ML                     0.240             0.002      0.990 0.387
                     1% ML OR rules                     0.313             0.005      0.983 0.475
                     5%          ML                     0.605             0.044      0.926 0.731
                     5% ML OR rules                     0.633             0.046      0.926 0.752
                    10%          ML                     0.805             0.094      0.885 0.843
                    10% ML OR rules                     0.823             0.096      0.885 0.853


## 11. What causes the false alarms?
Words that push legit (Enron) emails toward "phishing" most, at the 5% false-alarm threshold.

In [12]:
thr5 = np.quantile(p[val_idx][yv == 0], 0.95)
fp = np.where((p >= thr5) & (y_test == 0))[0]
print(f"False alarms at 5% threshold: {len(fp)} of {(y_test == 0).sum()} legit test emails")
contrib = X_test[fp].multiply(mdl.coef_[0]).sum(axis=0).A1           # each column's total push toward phishing
print(np.array(FULL_FEATURE_NAMES)[np.argsort(contrib)[::-1][:25]])

False alarms at 5% threshold: 139 of 2958 legit test emails
['sender_local_len_log' 'sender_is_receiver' 'exclaim_count_log'
 'sender_domain_len' 'TLD_com' 'upper_ratio' 'subject_upper_ratio'
 'body__click' 'kw_urgency_log' 'subj__information' 'subj__password'
 'subj__emailtoken' 'body__agreement' 'subj__notification' 'body__market'
 'body__website' 'u_ip' 'subj__transfer' 'subj__request' 'subj__notice'
 'body__request' 'body__financial' 'body__trading' 'body__love'
 'body__form']
